# S3 — Retrieval grounding across ecosystems: the analysis, regenerated

*Legacy id 13_1 (File A §10 Phase 14).* This notebook regenerates WP-8's signed `analyze_experiment` tables from the replication export alone and checks every file against the signed copy.

- **Recomputed:** correctness, which is a program (no LLM), from each generation in `runs/*/items.jsonl` and the frozen labelled set.
- **Read as recorded:** the judge's faithfulness and precision@k verdicts, from `runs/analysis/metrics.csv`. A judge call is not reproducible.
- **The judge is not validated:** WP-9's Cohen's kappa is **0.134** (File C §3.4.1 needs ≥ 0.50). Its faithfulness verdicts are not used for S3's claims: faithfulness comes from the 50 human labels (decisions §15.3, §5 below). Correctness does not depend on the judge.
- **TARGET hands over most answers** (decisions §13.2, §13.13): every condition sees each advisory's fixed version and the deprecation sentence. The table of items where it does not is the comparison copying cannot win.
- PyPI retrieval is mostly READMEs (decisions §13.14 (4)), a threat to validity for the PyPI arm and RQ3.

In [1]:
import os
import sys
from pathlib import Path

# The repository root, wherever Jupyter was started from inside it.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "backend" / "manage.py").exists())
sys.path.insert(0, str(ROOT / "backend"))
# Exports only, never a database: this settings module's database backend
# raises on the first query (config/settings/offline.py).
os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings.offline"

import django

django.setup()

from IPython.display import Image, Markdown, display

from apps.research import replication

EXPORT = replication.find_export(ROOT)  # or set RV_EXPORTS
# Regenerated files go here, never over the export.
WORK = Path(os.environ.get("RV_NOTEBOOK_WORK", ROOT / "research_data" / "notebook_work")) / "13_1_analysis"


def shown(path):
    """A path as the reader's checkout would print it."""
    path = Path(path)
    return path.relative_to(ROOT).as_posix() if path.is_relative_to(ROOT) else str(path)


print("export:", shown(EXPORT))

export: research_data/exports


## 1. The export is what was exported

In [2]:
check = replication.verify_export(EXPORT)
display(Markdown(replication.manifest_summary(check.manifest)))
assert check.ok, f"changed: {check.mismatched}; missing: {check.missing}"
print(f"{check.checked} files match the sha256 the export recorded")

| | |
|---|---|
| Exported | 2026-10-10T15:53:12Z |
| At commit | `49bde9054892` (local changes) |
| Database | postgresql `repovitals_research` |
| Sources | corpus_scan |
| `scan_history` | 914 rows (2026-10-07 914) |
| `dependency_history` | 29494 rows |
| `agent_traces` | 0 rows |
| Files | 52, each with its sha256 |

52 files match the sha256 the export recorded


## 2. Correctness, recomputed

Every (item, condition) record is measured again with `metrics.measure` and its deterministic columns compared with the signed `metrics.csv`.

In [3]:
s3 = replication.s3_dataset(EXPORT)
runs = ", ".join(f"{c} `{m['run_id']}`" for c, m in sorted(s3.dataset.runs.items()))
display(Markdown(f"{len(s3.dataset.rows)} item-condition records from {runs}."))
print(f"deterministic columns that disagree with the signed metrics.csv: {len(s3.disagreements)}")
assert not s3.disagreements, s3.disagreements[:10]

450 item-condition records from A `A_75246f3b_openai-gpt-oss-120b`, B `B_75246f3b_openai-gpt-oss-120b`, C `C_75246f3b_openai-gpt-oss-120b`.

deterministic columns that disagree with the signed metrics.csv: 0


## 3. The tables, regenerated

Rendered by `analysis.render`, the function `analyze_experiment` calls, with the bootstrap size and seed the signed `tables.md` states.

In [4]:
bootstrap, seed = replication.analysis_parameters(EXPORT)
replication.s3_tables(s3.dataset, WORK / "analysis", bootstrap=bootstrap, seed=seed)
results = replication.compare_folders(
    WORK / "analysis", EXPORT / replication.ANALYSIS_DIR, replication.s3_table_files(EXPORT)
)
display(Markdown(replication.comparison_table(results)))
assert all(result.identical for result in results), [r.differences for r in results if not r.identical]

| File | Against the signed copy |
|---|---|
| `calibration.csv` | **identical** |
| `correctness.csv` | **identical** |
| `correctness_not_given.csv` | **identical** |
| `faithfulness.csv` | **identical** |
| `metrics.csv` | **identical** |
| `paired_tests.csv` | **identical** |
| `precision.csv` | **identical** |
| `tables.md` | **identical** |

In [5]:
display(Markdown((WORK / "analysis" / "tables.md").read_text(encoding="utf-8")))

# S3 experiment analysis

Generated by `manage.py analyze_experiment` (§10 Phase 13). File C §3.4's prespecified analyses only; anything else is exploratory. Intervals: 95% bootstrap over items, 2000 resamples, seed 42.

## Runs

| Condition | Run | Generator | Items | Completed | Failed | Judged |
|---|---|---|---|---|---|---|
| A | `A_75246f3b_openai-gpt-oss-120b` | openai/gpt-oss-120b | 150 | 150 | 0 | 150 |
| B | `B_75246f3b_openai-gpt-oss-120b` | openai/gpt-oss-120b | 150 | 150 | 0 | 150 |
| C | `C_75246f3b_openai-gpt-oss-120b` | openai/gpt-oss-120b | 150 | 150 | 0 | 150 |

## Correctness (deterministic)

| Condition | Ecosystem | Case type | n | Correct | Rate | 95% interval |
|---|---|---|---|---|---|---|
| A | npm | cve_fix | 37 | 34 | 0.919 | [0.811, 1.000] |
| A | npm | deprecation_replacement | 38 | 17 | 0.447 | [0.289, 0.605] |
| A | npm | all | 75 | 51 | 0.680 | [0.573, 0.787] |
| A | pypi | cve_fix | 75 | 73 | 0.973 | [0.933, 1.000] |
| A | pypi | all | 75 | 73 | 0.973 | [0.933, 1.000] |
| B | npm | cve_fix | 37 | 33 | 0.892 | [0.784, 0.973] |
| B | npm | deprecation_replacement | 38 | 23 | 0.605 | [0.447, 0.763] |
| B | npm | all | 75 | 56 | 0.747 | [0.640, 0.853] |
| B | pypi | cve_fix | 75 | 66 | 0.880 | [0.800, 0.947] |
| B | pypi | all | 75 | 66 | 0.880 | [0.800, 0.947] |
| C | npm | cve_fix | 37 | 24 | 0.649 | [0.486, 0.784] |
| C | npm | deprecation_replacement | 38 | 33 | 0.868 | [0.737, 0.974] |
| C | npm | all | 75 | 57 | 0.760 | [0.667, 0.853] |
| C | pypi | cve_fix | 75 | 45 | 0.600 | [0.493, 0.707] |
| C | pypi | all | 75 | 45 | 0.600 | [0.493, 0.707] |

## Correctness where TARGET does not already show the answer

Every condition is shown each advisory's fixed version and the deprecation sentence (decisions §13.2). This table keeps only the items where no value shown would be correct if copied — a `cve_fix` item whose shown fixes are each still affected by another advisory. It is the comparison where retrieval has something to add (decisions §13.13); its paired tests are a Holm family of their own below.

Items: 150; answer given 128, not given 22.

| Condition | Ecosystem | n | Correct | Rate | 95% interval |
|---|---|---|---|---|---|
| A | npm | 8 | 8 | 1.000 | [1.000, 1.000] |
| A | pypi | 14 | 13 | 0.929 | [0.786, 1.000] |
| A | all | 22 | 21 | 0.955 | [0.864, 1.000] |
| B | npm | 8 | 7 | 0.875 | [0.625, 1.000] |
| B | pypi | 14 | 10 | 0.714 | [0.429, 0.929] |
| B | all | 22 | 17 | 0.773 | [0.591, 0.955] |
| C | npm | 8 | 3 | 0.375 | [0.125, 0.750] |
| C | pypi | 14 | 1 | 0.071 | [0.000, 0.214] |
| C | all | 22 | 4 | 0.182 | [0.045, 0.364] |

## Faithfulness (judged)

| Condition | Ecosystem | n | faithful | minor | major | major rate |
|---|---|---|---|---|---|---|
| A | npm | 75 | 71 | 3 | 1 | 0.013 |
| A | pypi | 75 | 74 | 1 | 0 | 0.000 |
| B | npm | 75 | 71 | 0 | 4 | 0.053 |
| B | pypi | 75 | 70 | 5 | 0 | 0.000 |
| C | npm | 75 | 69 | 1 | 5 | 0.067 |
| C | pypi | 75 | 63 | 5 | 7 | 0.093 |

`major_unsupported` — a hallucinated core claim — is the practically fatal category and has its own column (File C §3.4.3).

## Retrieval precision@k (judged; B, C, D)

| Condition | Ecosystem | n | Mean precision@k | 95% interval |
|---|---|---|---|---|
| B | npm | 70 | 0.030 | [0.014, 0.051] |
| B | pypi | 72 | 0.014 | [0.003, 0.028] |
| C | npm | 71 | 0.036 | [0.014, 0.061] |
| C | pypi | 72 | 0.014 | [0.003, 0.031] |

## Paired tests

Correctness: exact McNemar on discordant pairs; the odds ratio is (second only correct) / (first only correct). Faithfulness: Wilcoxon signed-rank on faithful=2 / minor=1 / major=0, effect = matched rank-biserial r, positive when the second condition is more faithful. Holm's correction runs within each metric's family (File C §3.4.5).

| Metric | Pair | Group | n (excluded) | Effect | p | p (Holm) |
|---|---|---|---|---|---|---|
| correctness | A -> B | all | 150 (0) | 16 vs 18 discordant; OR 0.89; rate diff -0.013 [-0.087, 0.067] | 0.8642 | 1.0000 |
| correctness | A -> B | npm | 75 (0) | 15 vs 10 discordant; OR 1.50; rate diff +0.067 [-0.067, 0.200] | 0.4244 | 1.0000 |
| correctness | A -> B | pypi | 75 (0) | 1 vs 8 discordant; OR 0.12; rate diff -0.093 [-0.173, -0.027] | 0.0391 | 0.1953 |
| correctness | B -> C | all | 150 (0) | 17 vs 37 discordant; OR 0.46; rate diff -0.133 [-0.227, -0.040] | 0.0091 | 0.0544 |
| correctness | B -> C | npm | 75 (0) | 14 vs 13 discordant; OR 1.08; rate diff +0.013 [-0.120, 0.147] | 1.0000 | 1.0000 |
| correctness | B -> C | pypi | 75 (0) | 3 vs 24 discordant; OR 0.12; rate diff -0.280 [-0.400, -0.160] | 0.0000 | 0.0004 |
| correctness | A -> C | all | 150 (0) | 20 vs 42 discordant; OR 0.48; rate diff -0.147 [-0.247, -0.047] | 0.0071 | 0.0500 |
| correctness | A -> C | npm | 75 (0) | 19 vs 13 discordant; OR 1.46; rate diff +0.080 [-0.067, 0.227] | 0.3771 | 1.0000 |
| correctness | A -> C | pypi | 75 (0) | 1 vs 29 discordant; OR 0.03; rate diff -0.373 [-0.493, -0.253] | 0.0000 | 0.0000 |
| correctness, answer not given | A -> B | all | 22 (0) | 0 vs 4 discordant; OR 0.00; rate diff -0.182 [-0.364, -0.045] | 0.1250 | 0.5000 |
| correctness, answer not given | A -> B | npm | 8 (0) | 0 vs 1 discordant; OR 0.00; rate diff -0.125 [-0.375, 0.000] | 1.0000 | 1.0000 |
| correctness, answer not given | A -> B | pypi | 14 (0) | 0 vs 3 discordant; OR 0.00; rate diff -0.214 [-0.429, 0.000] | 0.2500 | 0.5000 |
| correctness, answer not given | B -> C | all | 22 (0) | 0 vs 13 discordant; OR 0.00; rate diff -0.591 [-0.773, -0.364] | 0.0002 | 0.0020 |
| correctness, answer not given | B -> C | npm | 8 (0) | 0 vs 4 discordant; OR 0.00; rate diff -0.500 [-0.875, -0.125] | 0.1250 | 0.5000 |
| correctness, answer not given | B -> C | pypi | 14 (0) | 0 vs 9 discordant; OR 0.00; rate diff -0.643 [-0.857, -0.357] | 0.0039 | 0.0234 |
| correctness, answer not given | A -> C | all | 22 (0) | 0 vs 17 discordant; OR 0.00; rate diff -0.773 [-0.955, -0.591] | 0.0000 | 0.0001 |
| correctness, answer not given | A -> C | npm | 8 (0) | 0 vs 5 discordant; OR 0.00; rate diff -0.625 [-0.875, -0.250] | 0.0625 | 0.3125 |
| correctness, answer not given | A -> C | pypi | 14 (0) | 0 vs 12 discordant; OR 0.00; rate diff -0.857 [-1.000, -0.643] | 0.0005 | 0.0034 |
| faithfulness | A -> B | all | 150 (0) | r = -0.390 (normal, 14 nonzero) | 0.1932 | 0.7728 |
| faithfulness | A -> B | npm | 75 (0) | r = -0.333 (normal, 8 nonzero) | 0.4273 | 0.8546 |
| faithfulness | A -> B | pypi | 75 (0) | r = -0.667 (normal, 6 nonzero) | 0.1294 | 0.6472 |
| faithfulness | B -> C | all | 150 (0) | r = -0.542 (normal, 22 nonzero) | 0.0229 | 0.1376 |
| faithfulness | B -> C | npm | 75 (0) | r = -0.238 (normal, 6 nonzero) | 0.6567 | 0.8546 |
| faithfulness | B -> C | pypi | 75 (0) | r = -0.676 (normal, 16 nonzero) | 0.0149 | 0.1041 |
| faithfulness | A -> C | all | 150 (0) | r = -0.739 (normal, 22 nonzero) | 0.0019 | 0.0167 |
| faithfulness | A -> C | npm | 75 (0) | r = -0.467 (normal, 9 nonzero) | 0.2235 | 0.7728 |
| faithfulness | A -> C | pypi | 75 (0) | r = -0.923 (normal, 13 nonzero) | 0.0028 | 0.0227 |

## RQ3: npm vs PyPI

Within the `cve_fix` stratum first — the controlled ecosystem head-to-head (File C §3.4.6, L7) — then over the full set, where the two ecosystems contribute different case mixes and the comparison carries that confound.

| Stratum | Condition | npm n | npm rate | PyPI n | PyPI rate | npm - PyPI | 95% interval |
|---|---|---|---|---|---|---|---|
| cve_fix | A | 37 | 0.919 | 75 | 0.973 | -0.054 | [-0.164, 0.029] |
| cve_fix | B | 37 | 0.892 | 75 | 0.880 | +0.012 | [-0.118, 0.133] |
| cve_fix | C | 37 | 0.649 | 75 | 0.600 | +0.049 | [-0.138, 0.252] |
| full set (case-mix confounded) | A | 75 | 0.680 | 75 | 0.973 | -0.293 | [-0.406, -0.180] |
| full set (case-mix confounded) | B | 75 | 0.747 | 75 | 0.880 | -0.133 | [-0.252, -0.011] |
| full set (case-mix confounded) | C | 75 | 0.760 | 75 | 0.600 | +0.160 | [0.009, 0.303] |

## Insufficient-information calibration (File C §3.4.7)

How often each gated condition said it could not recommend, against its own grounding flag.

| Condition | Ecosystem | Grounding | n | Declared insufficient | Rate |
|---|---|---|---|---|---|
| A | npm | sufficient | 75 | 17 | 0.227 |
| A | pypi | sufficient | 75 | 13 | 0.173 |
| B | npm | sufficient | 75 | 1 | 0.013 |
| B | pypi | sufficient | 75 | 0 | 0.000 |
| C | npm | sufficient | 66 | 0 | 0.000 |
| C | npm | low | 9 | 9 | 1.000 |
| C | pypi | sufficient | 72 | 0 | 0.000 |
| C | pypi | low | 3 | 3 | 1.000 |

## Reading these tables

- Every result is reported at face value with its interval; a null is a finding (File C §3.4.8).
- Correctness counts the structured fix only, and only against the advisories on the resolved version (decisions §13).
- TARGET shows every condition the fixed version and the deprecation sentence the answers were extracted from, so correctness partly measures use of given facts (decisions §13.2); the answer-not-given table is the comparison that does not (§13.13).
- Faithfulness is an LLM's verdict; read it with WP-9's kappa (File C L5).


## 4. WP-9: the judge's validation

The answer key that maps `ITEM-001`… back to a run, an item and the judge's verdict is rebuilt from the export: the packet's sample is a seeded function of the runs, so `replication.wp9_key` re-draws it and checks every item against the committed packet verbatim, refusing on any difference (decisions §15.3). The judge's verdicts are the recorded ones. With the original key, set `RV_WP9_KEY` to its path instead.

In [6]:
import json

from apps.research.experiment import judge_validation

key_path = os.environ.get("RV_WP9_KEY")
if key_path:
    key = json.loads(Path(key_path).read_text(encoding="utf-8"))["items"]
    print("key: RV_WP9_KEY")
else:
    try:
        key = replication.wp9_key(EXPORT)
        print(f"key: rebuilt from the export, {len(key)} items, every one verbatim in the packet")
    except judge_validation.ValidationPacketError as refusal:
        key = None
        print(f"The key could not be rebuilt from this export: {refusal}")
if key is not None:
    labels = judge_validation.read_labels(EXPORT / replication.WP9_LABELS, key)
    wp9 = judge_validation.kappa(labels, key)
    display(Markdown(judge_validation.kappa_report(wp9)))

key: rebuilt from the export, 50 items, every one verbatim in the packet


# WP-9: judge validation

- Items: 50
- Cohen's kappa (human vs judge): **0.134**
- Linearly weighted kappa (ordinal): 0.186
- Raw agreement: 68.0%
- **Decision:** kappa < 0.50: tighten the rubric, re-judge and re-validate before any S3 claim

## Confusion (rows: human, columns: judge)

| | faithful | minor_unsupported | major_unsupported |
|---|---|---|---|
| faithful | 33 | 1 | 0 |
| minor_unsupported | 8 | 1 | 1 |
| major_unsupported | 4 | 2 | 0 |

## Disagreements

| Item | Human | Judge | Condition | Ecosystem | Human note | Judge note |
|---|---|---|---|---|---|---|
| ITEM-001 | minor_unsupported | faithful | A | pypi | 'Updating the dependency to the latest version will address all known vulnerabilities' is not supported: CVE-2026-5241 has no recorded fixed version and only 5 of 44 advisory records are detailed; target 5.19.0 is the latest release in the measured data. |  |
| ITEM-003 | major_unsupported | minor_unsupported | B | pypi | 'The upgrade does not change its API surface' is not in the measured data or passages (they only say h11 is a toolkit); treated as core because it asserts the upgrade is non-breaking. Target 0.16.0 is supported. |  |
| ITEM-006 | minor_unsupported | faithful | A | pypi | 'We cannot point to a specific release that addresses these issues' contradicts the measured data (fixed in 1.0.1 and 1.5.0); target 1.9.1 is the latest release and above both fixes. |  |
| ITEM-009 | minor_unsupported | faithful | C | npm | 'A direct upgrade to 7.23.2 should be safe' is not in the evidence (nothing shown covers 7.x upgrades); target 7.23.2 and the cited Babel 8 breaking change are supported. |  |
| ITEM-014 | minor_unsupported | major_unsupported | C | pypi | Several peripheral claims unsupported: 'added encoding keyword support' is presented as part of this upgrade but the passage puts it in a pre-0.1.6 release, not 0.5.0, and 'the upgrade should be straightforward' is not stated in the evidence; target 0.5.0, the DoS fix and the Python 3.5-3.7 drop are supported. |  |
| ITEM-016 | minor_unsupported | faithful | B | pypi | Several peripheral claims unsupported: 'will eliminate the known vulnerabilities' goes beyond the evidence (101 advisory records, only 5 detailed), and the README passage only links the release notes and changelog without saying the security fixes are documented there; target 12.3.0 is the latest release and above every detailed fix. |  |
| ITEM-020 | major_unsupported | faithful | B | pypi | '(Or at least 0.5.0) removes all known vulnerabilities' contradicts the measured data (CVE-2026-59893 is fixed only in 0.6.0); target 0.6.0, the latest release, is supported. |  |
| ITEM-022 | faithful | minor_unsupported | C | pypi |  |  |
| ITEM-025 | major_unsupported | faithful | A | npm | Says 'details about a rename or migration path are not available' and recommends staying on flag-icon-css 4.1.7, but the deprecation message names the rename to flag-icons; the core recommendation (no replacement) contradicts the measured data. |  |
| ITEM-026 | major_unsupported | faithful | A | npm | 'No source information confirming a ... successor' is contradicted by the deprecation message, which names stylelint-config-suitcss as the replacement; the core recommendation (investigate/remove, no replacement) is not what the measured data supports. |  |
| ITEM-034 | minor_unsupported | faithful | C | npm | 'No additional options ... are documented' is contradicted by passage 0d203f0dd0ca, which deprecates the spec and loose options of transform-private-methods; the replacement itself is named in the deprecation message. |  |
| ITEM-038 | minor_unsupported | faithful | B | npm | 'Appears in Babel's changelog as the successor' misdescribes passage 24ffd7fc7485, which only lists the plugin under a Breaking Change heading; the replacement is named in the deprecation message and 'latest = 7.20.7' is in the measured data. |  |
| ITEM-040 | minor_unsupported | faithful | A | npm | 'The advisory data lists fixed versions starting at 1.16.0' misstates the measured data (fixes range from 1.13.5 to 1.16.0); target 1.20.0 is the latest release and above every detailed fix. |  |
| ITEM-042 | minor_unsupported | faithful | C | pypi | 'The upgrade is safe for most codebases' is not in the evidence (no passage says the 2.20.0-to-2.31.0 upgrade is safe); target 2.31.0, the Proxy-Authorization fix and the credential-rotation advice are supported. |  |
| ITEM-043 | major_unsupported | faithful | B | npm | 'At least 9.0.5 ... will resolve the known vulnerabilities' contradicts the measured data (CVE-2026-35515 is fixed only in 11.1.18); target 12.1.2 itself is the latest release. |  |
| ITEM-048 | major_unsupported | minor_unsupported | B | pypi | 'This change does not require code modifications for basic encode/decode usage' (a 1.x-to-2.x non-breaking claim) is not in the measured data or passages; target 2.14.0 and the latest release 2.15.1 are supported. |  |


## 5. The S3 supplement: faithfulness from the human labels, the interaction, the examples

`apps.research.experiment.supplement` computes what WP-8's tables do not:

- **faithfulness from WP-9's 50 human labels**, by condition and ecosystem, beside the judge's verdicts on the same items — decisions §15.3's choice between §13.17's two options, since the judge's kappa bars its own verdicts;
- **the condition × ecosystem interaction** (File C §3.4.6), within `cve_fix` first;
- **the qualitative slice** (File C §3.4.9), one example per kind, each chosen by a fixed rule.

The paper's copy is `papers/s3/analysis/`; when this export holds the data it was computed from, every regenerated file is compared with it.

In [7]:
from apps.research.experiment import runner
from apps.research.experiment import supplement as s3_supplement

if key is None:
    print("No key, so no human-label analysis on this export.")
else:
    items, _ = runner.load_items(EXPORT / replication.LABELLED_SET)
    records = {
        (record["item_id"], record["condition"]): record
        for run_dir in replication.run_directories(EXPORT)
        for record in runner.latest_records(run_dir / runner.ITEMS_FILENAME).values()
    }
    found = s3_supplement.run(
        s3.dataset.rows,
        records,
        {item["item_id"]: item for item in items},
        labels,
        key,
        seed=seed,
        iterations=bootstrap,
        inputs_digest=s3_supplement.input_digests(EXPORT),
    )
    s3_supplement.write(found, WORK / "s3_supplement")
    PAPER = ROOT / "papers" / "s3" / "analysis"
    if s3_supplement.same_inputs(EXPORT, PAPER):
        results = replication.compare_folders(WORK / "s3_supplement", PAPER, list(s3_supplement.FILES))
        display(Markdown(replication.comparison_table(results)))
        assert all(result.identical for result in results), [r.differences for r in results if not r.identical]
    else:
        print(f"This export is not the data {shown(PAPER)} was computed from: regenerated, not compared.")
    display(Markdown((WORK / "s3_supplement" / "supplement.md").read_text(encoding="utf-8")))

| File | Against the signed copy |
|---|---|
| `inputs.json` | **identical** |
| `supplement.md` | **identical** |
| `human_faithfulness.csv` | **identical** |
| `interaction.csv` | **identical** |
| `examples.md` | **identical** |
| `wp9_kappa.md` | **identical** |
| `judge_validation_key_rebuilt.json` | **identical** |

# S3 supplementary analyses

Generated by `apps.research.experiment.supplement` from the replication export, seed 42, 2000 bootstrap resamples.

## 1. The judge, recomputed (File C §3.4.1)

WP-9's answer key, rebuilt from the runs and checked against the packet verbatim: Cohen's kappa **0.134**, linearly weighted 0.186, raw agreement 68% over 50 items. Below 0.50, so the judge's faithfulness verdicts are not used for S3's claims; faithfulness comes from the human labels (decisions §15.3).

## 2. Faithfulness from the human labels

One human label per sampled generation, stratified by condition x ecosystem and not paired across conditions, so the groups are compared descriptively. Intervals are Wilson 95%.

| Condition | Ecosystem | n | Human: faithful / minor / major | Unsupported (minor or major), 95% Wilson | Major, 95% Wilson | Judge on the same items: faithful / minor / major |
|---|---|---|---|---|---|---|
| A | npm | 9 | 6 / 1 / 2 | 0.33 [0.12, 0.65] | 0.22 [0.06, 0.55] | 9 / 0 / 0 |
| A | pypi | 9 | 7 / 2 / 0 | 0.22 [0.06, 0.55] | 0.00 [0.00, 0.30] | 9 / 0 / 0 |
| A | all | 18 | 13 / 3 / 2 | 0.28 [0.12, 0.51] | 0.11 [0.03, 0.33] | 18 / 0 / 0 |
| B | npm | 8 | 6 / 1 / 1 | 0.25 [0.07, 0.59] | 0.12 [0.02, 0.47] | 8 / 0 / 0 |
| B | pypi | 8 | 4 / 1 / 3 | 0.50 [0.22, 0.78] | 0.38 [0.14, 0.69] | 6 / 2 / 0 |
| B | all | 16 | 10 / 2 / 4 | 0.38 [0.18, 0.61] | 0.25 [0.10, 0.49] | 14 / 2 / 0 |
| C | npm | 8 | 5 / 3 / 0 | 0.38 [0.14, 0.69] | 0.00 [0.00, 0.32] | 7 / 1 / 0 |
| C | pypi | 8 | 6 / 2 / 0 | 0.25 [0.07, 0.59] | 0.00 [0.00, 0.32] | 6 / 1 / 1 |
| C | all | 16 | 11 / 5 / 0 | 0.31 [0.14, 0.56] | 0.00 [0.00, 0.19] | 13 / 2 / 1 |
| all | all | 50 | 34 / 10 / 6 | 0.32 [0.21, 0.46] | 0.12 [0.06, 0.24] | 45 / 4 / 1 |

## 3. Condition x ecosystem interaction (File C §3.4.6)

Effect = paired change in correctness rate from the first condition to the second, per ecosystem; difference = npm effect - PyPI effect. Positive: the change helps npm more (or hurts PyPI more). Interval: 95% bootstrap resampling items within each ecosystem.

| Stratum | Pair | npm n | npm effect | PyPI n | PyPI effect | Difference | 95% interval |
|---|---|---|---|---|---|---|---|
| cve_fix | A -> B | 37 | -0.027 | 75 | -0.093 | +0.066 | [-0.095, +0.228] |
| cve_fix | B -> C | 37 | -0.243 | 75 | -0.280 | +0.037 | [-0.179, +0.251] |
| cve_fix | A -> C | 37 | -0.270 | 75 | -0.373 | +0.103 | [-0.113, +0.317] |
| all | A -> B | 75 | +0.067 | 75 | -0.093 | +0.160 | [+0.013, +0.320] |
| all | B -> C | 75 | +0.013 | 75 | -0.280 | +0.293 | [+0.107, +0.467] |
| all | A -> C | 75 | +0.080 | 75 | -0.373 | +0.453 | [+0.280, +0.640] |

## 4. The qualitative slice

8 examples, one per kind: `examples.md`.

| # | Kind | Item | Condition | Ecosystem |
|---|---|---|---|---|
| 1 | grounded and correct | `S3-3dc2ebde9f` | C | pypi |
| 2 | grounded but wrong | `S3-07e91951e7` | B | npm |
| 3 | honest abstention | `S3-211276731e` | C | npm |
| 4 | unsupported core claim | `S3-04ab73079a` | B | pypi |
| 5 | overclaim on coverage | `S3-45c2ef2c7b` | B | pypi |
| 6 | missed successor | `S3-30835ec6e2` | A | npm |
| 7 | retrieval made it worse | `S3-03551af2a3` | C | pypi |
| 8 | README-only retrieval | `S3-025ccd1209` | B | pypi |


## 6. Exploring

One row per (item, condition), as the analysis measured it. Anything built from here is exploratory unless File C §3.4 prespecified it.

In [8]:
import pandas as pd

frame = pd.DataFrame([row.as_json() for row in s3.dataset.rows])
frame.pivot_table(index=["ecosystem", "case_type"], columns="condition", values="correct", aggfunc="mean").round(3)

condition                              A      B      C
ecosystem case_type                                   
npm       cve_fix                  0.919  0.892  0.649
          deprecation_replacement  0.447  0.605  0.868
pypi      cve_fix                  0.973  0.880  0.600